# Lab 06 - Deploying a model with FastAPI
AI-101L, Arham. The Titanic survival SVC from Lab 05 data is served as a REST API. Everything below runs the real app in-process (`TestClient`) and then a real `uvicorn` server.

In [1]:
import sys, json, time, statistics, subprocess, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..') if False else None
from pathlib import Path
print(json.dumps(json.load(open('artifacts/training_summary.json')), indent=2))

{
  "rows": 889,
  "train_rows": 711,
  "test_rows": 178,
  "test_accuracy": 0.7697,
  "baseline_accuracy": 0.618,
  "possible_requests": 72,
  "pickle_pipeline_agreement": 72
}


## 1. Valid request, all three prediction routes

In [2]:
from fastapi.testclient import TestClient
from app.main import app
client = TestClient(app).__enter__()
good = {'PClass':'First','Gender':'Female','Sibling':'Zero','Embarked':'Cherbourg'}
for path in ['/predict','/predict/pipeline','/predict/lenient']:
    r = client.post(path, json=good)
    print(path, r.status_code, r.json())

/predict 200 {'predicted_label': 1, 'prediction': 'SURVIVED', 'confidence': 0.8323}
/predict/pipeline 200 {'predicted_label': 1, 'prediction': 'SURVIVED', 'confidence': 0.8323}
/predict/lenient 200 {'predicted_label': 1, 'prediction': 'SURVIVED', 'confidence': 0.8323, 'unknown_fields': []}


## 2. Error handling: 400 (unknown category) vs 422 (malformed body)

In [3]:
r = client.post('/predict', json={**good,'Embarked':'Atlantis'}); print(r.status_code, r.json())
r = client.post('/predict', json={k:v for k,v in good.items() if k!='Gender'}); print(r.status_code, r.json()['detail'][0]['msg'])
r = client.post('/predict', json={**good,'Sibling':1}); print(r.status_code, r.json()['detail'][0]['msg'])
r = client.post('/predict/lenient', json={**good,'Embarked':'Atlantis'}); print(r.status_code, r.json())

400 {'detail': "Unknown value 'Atlantis' for Embarked. Allowed: ['Cherbourg', 'Queenstown', 'Southampton']"}
422 Field required
422 Input should be a valid string
200 {'predicted_label': 1, 'prediction': 'SURVIVED', 'confidence': 0.6756, 'unknown_fields': ['Embarked']}


## 3. Who survives? All 72 possible passengers

In [4]:
import itertools, pandas as pd
rows=[]
for p,g,s,e in itertools.product(['First','Second','Third'],['Female','Male'],['Zero','One','Two','Three'],['Cherbourg','Queenstown','Southampton']):
    b=client.post('/predict',json=dict(PClass=p,Gender=g,Sibling=s,Embarked=e)).json(); rows.append((p,g,b['predicted_label']))
df=pd.DataFrame(rows,columns=['PClass','Gender','survived']); print(df.groupby(['PClass','Gender']).survived.mean().unstack())

Gender  Female      Male
PClass                  
First     1.00  0.166667
Second    1.00  0.000000
Third     0.75  0.000000


## 4. Latency
Two measurements: in-process (no network) and a real uvicorn server over HTTP, 200 requests each.

In [5]:
def timeit(fn, n=200):
    t=[]
    for _ in range(n):
        s=time.perf_counter(); fn(); t.append((time.perf_counter()-s)*1000)
    t.sort(); return round(statistics.mean(t),2), round(t[int(.95*n)],2)
print('in-process  /predict  mean, p95 ms:', timeit(lambda: client.post('/predict', json=good)))
print('in-process  /pipeline mean, p95 ms:', timeit(lambda: client.post('/predict/pipeline', json=good)))

in-process  /predict  mean, p95 ms: (8.48, 11.17)


in-process  /pipeline mean, p95 ms: (14.11, 18.88)


In [6]:
import httpx
server = subprocess.Popen([sys.executable,'-m','uvicorn','app.main:app','--port','8123','--log-level','warning'])
for _ in range(50):
    try:
        httpx.get('http://127.0.0.1:8123/health'); break
    except Exception: time.sleep(.2)
with httpx.Client() as h:
    print('uvicorn /predict mean, p95 ms:', timeit(lambda: h.post('http://127.0.0.1:8123/predict', json=good)))
    print(h.get('http://127.0.0.1:8123/metrics').text.split('# HELP api_request_seconds')[0][-400:])
server.terminate()

uvicorn /predict mean, p95 ms: (8.48, 11.57)
_total counter
api_requests_total{method="GET",path="/health",status="200"} 1.0
api_requests_total{method="POST",path="/predict",status="200"} 200.0
# HELP api_requests_created Requests handled
# TYPE api_requests_created gauge
api_requests_created{method="GET",path="/health",status="200"} 1.79388121570722e+09
api_requests_created{method="POST",path="/predict",status="200"} 1.7938812157733347e+09



## 5. Takeaways
- Loading artefacts once in `lifespan` keeps per-request time to a few milliseconds; the SVC inference itself is tiny.
- Plain `str` fields keep 422 for malformed bodies and 400 for well-formed but unknown categories.
- The pipeline route removes the five-file encoder bookkeeping and gave identical answers on all 72 inputs.